# 00 · Preparación de datos

Produce los archivos limpios que consumen los notebooks 01, 02 y 03.

**Salidas:**
- `data/eph_ocupados_2017plus.parquet` — ocupados en EPH 2017–2025 con las 6 variables del LCA y los IDs `CODUSU/NRO_HOGAR/COMPONENTE`.
- `data/engh_ocupados.parquet` — ocupados de la ENGH 2017/18 con sus 5 variables comunes con la EPH.
- `data/engh_hogares.parquet` — los 21.547 hogares con `gc_01..gc_12` y los `share_*` derivados.

**Decisiones explícitas (ver README §3):**
- Filtrado a ocupados (`ESTADO=='Ocupado'` en EPH; `estado==1` en ENGH).
- Edad discretizada en 5 buckets idénticos en ambas encuestas.
- `NIVEL_ED` reducido a 3 niveles.
- `PP07H` con tres categorías (`Formal`, `Informal`, `NoAplica` para cuenta propia/patrones).
- Regiones ENGH (1–6) mapeadas con codebook estándar INDEC.

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os

EPH_PANEL = "../data/panel_eph.parquet"
ENGH_DIR  = "../../ENGH_2017-18"
OUT       = "data"
os.makedirs(OUT, exist_ok=True)

## EPH — ocupados 2017+

In [ ]:
cols = ["ANO4","TRIMESTRE","REGION","PONDERA",
        "CH04","CH06","NIVEL_ED","ESTADO","CAT_OCUP","PP07H",
        "P21","P47T","NRO_HOGAR","COMPONENTE","CODUSU"]
table = pq.read_table(EPH_PANEL, columns=cols,
                      filters=[("ANO4",">=",2017), ("ESTADO","=","Ocupado")])
df = table.to_pandas()

def edad_grp(e):
    if pd.isna(e): return None
    e = int(e)
    if e < 25: return "<25"
    if e < 35: return "25-34"
    if e < 50: return "35-49"
    if e < 65: return "50-64"
    return "65+"
df["EDAD_GRP"] = df["CH06"].apply(edad_grp)

ed_map = {
    "Sin Instruccion": "Hasta_primario",
    "Primario Incompleto": "Hasta_primario",
    "Primario Completo": "Hasta_primario",
    "Secundario Incompleto": "Secundario",
    "Secundario Completo": "Secundario",
    "Superior o Universitario Incompleto": "Superior",
    "Superior o Universitario Completo": "Superior",
}
df["NIVEL_ED_3"] = df["NIVEL_ED"].map(ed_map)
df["FORMAL"] = df["PP07H"].map({"Si":"Formal","No":"Informal"}).fillna("NoAplica")

keep = ["ANO4","TRIMESTRE","REGION","PONDERA",
        "CH04","EDAD_GRP","NIVEL_ED_3","CAT_OCUP","FORMAL",
        "P21","P47T","NRO_HOGAR","COMPONENTE","CODUSU"]
df = df[keep].rename(columns={"CH04":"SEXO"}).dropna(
    subset=["SEXO","EDAD_GRP","NIVEL_ED_3","CAT_OCUP","FORMAL","REGION"])
print("EPH ocupados 2017+:", df.shape)
df.to_parquet(f"{OUT}/eph_ocupados_2017plus.parquet", index=False)

## ENGH — ocupados (individuos) y hogares con shares

In [ ]:
cols = ["id","miembro","pondera","region","cp03","cp04","cp13",
        "niveled","estado","cat_ocup","ingtotp"]
p = pd.read_csv(f"{ENGH_DIR}/engho2018_personas.txt", sep="|",
                encoding="utf-8", usecols=cols, low_memory=False)
ocup = p[p["estado"]==1].copy()

# Codebook ENGH:  cp13 1=Varón 2=Mujer  |  niveled 1-2=prim, 3-4=sec, 5-6=sup
ocup["SEXO"]       = ocup["cp13"].map({1:"Varon", 2:"Mujer"})
ocup["EDAD_GRP"]   = ocup["cp03"].apply(edad_grp)
ocup["NIVEL_ED_3"] = ocup["niveled"].map({1:"Hasta_primario",2:"Hasta_primario",
                                          3:"Secundario",4:"Secundario",
                                          5:"Superior",6:"Superior"})
ocup["CAT_OCUP"]   = ocup["cat_ocup"].map({1:"Patron",2:"Cuenta Propia",
                                           3:"Obrero o empleado",
                                           4:"Trabajador familiar sin remuneracion"})
ocup["REGION"]     = ocup["region"].map({1:"Gran Buenos Aires",2:"Pampeana",
                                         3:"NOA",4:"NEA",5:"Cuyo",6:"Patagonia"})

ocup_clean = ocup[["id","miembro","pondera","SEXO","EDAD_GRP","NIVEL_ED_3",
                   "CAT_OCUP","REGION","ingtotp"]].dropna(
    subset=["SEXO","EDAD_GRP","NIVEL_ED_3","CAT_OCUP","REGION"])
print("ENGH ocupados:", ocup_clean.shape)
ocup_clean.to_parquet(f"{OUT}/engh_ocupados.parquet", index=False)

In [ ]:
hcols = ["id","pondera","region","jsexo","jedad_agrup","jniveled","jocupengh",
         "cantmiem","menor14","mayor65","regten","gastot",
         "gc_01","gc_02","gc_03","gc_04","gc_05","gc_06",
         "gc_07","gc_08","gc_09","gc_10","gc_11","gc_12",
         "ingtoth","decgaphr","decgapht"]
h = pd.read_csv(f"{ENGH_DIR}/engho2018_hogares.txt", sep="|",
                encoding="utf-8", usecols=hcols, low_memory=False)
for i in range(1,13):
    h[f"share_{i:02d}"] = h[f"gc_{i:02d}"] / h["gastot"]
print("Hogares:", h.shape)
print("Suma promedio de shares (debe ser 1.0):",
      h[[f"share_{i:02d}" for i in range(1,13)]].sum(axis=1).mean().round(4))
h.to_parquet(f"{OUT}/engh_hogares.parquet", index=False)